In [1]:
from pathlib import Path
import random
import shutil
import yaml
from ultralytics import YOLO
from typing import Dict
import torch

In [2]:
data_dir = Path("dataset")
image_dir = Path(data_dir, "images")
label_dir = Path(data_dir, "labels")

image_dir.mkdir(exist_ok=True)
label_dir.mkdir(exist_ok=True)

In [3]:
image_dir

PosixPath('dataset/images')

In [4]:
label_dir

PosixPath('dataset/labels')

In [ ]:
def split_dataset(image_dir: str | Path, label_dir: str | Path) -> Dict[str, Path]:
    """
    Split an image dataset and its corresponding YOLO labels into
    training, validation, and test sets.

    The dataset is split using the following proportions:

        - 80% training data
        - 10% validation data
        - 10% test data

    Images and their corresponding YOLO annotation files are moved
    together so that each image remains paired with its label.

    Args:
        image_dir (str | Path):
            Directory containing the original image files.

        label_dir (str | Path):
            Directory containing the corresponding YOLO `.txt`
            annotation files. Each label file must have the same
            filename stem as its image.

            Example:
                image:  car_001.jpg
                label:  car_001.txt

    Returns:
        Dict[str, Path]:
            A dictionary containing the paths to the newly created
            training, validation, and test image and label directories.

            Returned keys:
                - "train images dir"
                - "val images dir"
                - "test images dir"
                - "train label dir"
                - "val labels dir"
                - "test labels dir"

    Example:
        >>> split_dataset("images", "labels")
        {
            "train images dir": Path("images/train_images"),
            "val images dir": Path("images/val_images"),
            "test images dir": Path("images/test_images"),
            "train label dir": Path("labels/train_labels"),
            "val labels dir": Path("labels/val_labels"),
            "test labels dir": Path("labels/test_labels")
        }

    Notes:
        - A fixed random seed of 42 is used so that the same dataset
          produces the same split when the function is run on the
          original unsplit dataset.
        - The function physically moves files using `shutil.move()`.
        - Therefore, this function should not be repeatedly run on an
          already-split dataset.
    """

    # Convert the supplied directory paths into Path objects.
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    # Get all items currently inside the image directory.
    list_images = list(image_dir.iterdir())

    # ------------------------------------------------------------------
    # Create directories for training, validation, and test images.
    # ------------------------------------------------------------------

    train_images = Path(image_dir, "train_images")
    train_images.mkdir(exist_ok=True)

    val_images = Path(image_dir, "val_images")
    val_images.mkdir(exist_ok=True)

    test_images = Path(image_dir, "test_images")
    test_images.mkdir(exist_ok=True)

    # ------------------------------------------------------------------
    # Create directories for training, validation, and test labels.
    # ------------------------------------------------------------------

    train_labels = Path(label_dir, "train_labels")
    train_labels.mkdir(exist_ok=True)

    val_labels = Path(label_dir, "val_labels")
    val_labels.mkdir(exist_ok=True)

    test_labels = Path(label_dir, "test_labels")
    test_labels.mkdir(exist_ok=True)

    # ------------------------------------------------------------------
    # Shuffle the dataset before splitting.
    #
    # The seed makes the shuffle reproducible. This means that if the
    # exact same original dataset is used again, the same images will
    # be assigned to train, validation, and test.
    # ------------------------------------------------------------------

    random.seed(42)
    random.shuffle(list_images)

    # Calculate the number of images assigned to each split.
    train_size = int(len(list_images) * 0.80)
    val_size = int(len(list_images) * 0.10)

    # ------------------------------------------------------------------
    # Slice the shuffled image list into three separate datasets.
    #
    # Example with 100 images:
    #
    # train -> images 0 to 79
    # val   -> images 80 to 89
    # test  -> images 90 to 99
    # ------------------------------------------------------------------

    train_image_list = list_images[:train_size]

    val_image_list = list_images[
        train_size:train_size + val_size
    ]

    test_image_list = list_images[
        train_size + val_size:
    ]

    # ------------------------------------------------------------------
    # Move training images and their corresponding labels.
    # ------------------------------------------------------------------

    for image in train_image_list:

        # Construct the path to the label with the same filename stem.
        #
        # Example:
        #     image = car_001.jpg
        #     label = labels/car_001.txt
        #
        label = label_dir / f"{image.stem}.txt"

        # Move the image into the training image directory.
        shutil.move(
            image,
            train_images / image.name
        )

        # Move its corresponding label into the training label directory.
        shutil.move(
            label,
            train_labels / label.name
        )

    # ------------------------------------------------------------------
    # Move validation images and their corresponding labels.
    # ------------------------------------------------------------------

    for val_img in val_image_list:

        # Find the corresponding YOLO annotation file.
        val_lbl = label_dir / f"{val_img.stem}.txt"

        # Move the validation image.
        shutil.move(
            val_img,
            val_images / val_img.name
        )

        # Move its corresponding validation label.
        shutil.move(
            val_lbl,
            val_labels / val_lbl.name
        )

    # ------------------------------------------------------------------
    # Move test images and their corresponding labels.
    # ------------------------------------------------------------------

    for test_image in test_image_list:

        # Find the corresponding YOLO annotation file.
        test_label = label_dir / f"{test_image.stem}.txt"

        # Move the test image.
        shutil.move(
            test_image,
            test_images / test_image.name
        )

        # Move its corresponding test label.
        shutil.move(
            test_label,
            test_labels / test_label.name
        )

    # Return all of the newly created directories so that they can
    # easily be used later when creating the YOLO dataset configuration.
    return {
        "train images dir": train_images,
        "val images dir": val_images,
        "test images dir": test_images,
        "test labels dir": test_labels,
        "train label dir": train_labels,
        "val labels dir": val_labels,
    }

In [8]:
split_dataset(image_dir, label_dir)

{'train images dir': WindowsPath('dataset/images/train_images'),
 'val images dir': WindowsPath('dataset/images/val_images'),
 'test images dir': WindowsPath('dataset/images/test_images'),
 'test labels dir': WindowsPath('dataset/labels/test_labels'),
 'train label dir': WindowsPath('dataset/labels/train_labels'),
 'val labels dir': WindowsPath('dataset/labels/val_labels')}

In [5]:
classes = ['person',
 'bicycle',
 'car',
 'motorcycle',
 'airplane',
 'bus',
 'train',
 'truck',
 'boat',
 'traffic light',
 'fire hydrant',
 'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
 'carrot',
 'hot dog',
 'pizza',
 'donut',
 'cake',
 'chair',
 'couch',
 'potted plant',
 'bed',
 'dining table',
 'toilet',
 'tv',
 'laptop',
 'mouse',
 'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
 'sink',
 'refrigerator',
 'book',
 'clock',
 'vase',
 'scissors',
 'teddy bear',
 'hair drier',
 'toothbrush',
 'door',
 'stairs',
 'window']


In [6]:
metadata = {
    'path': str(data_dir.absolute()),
    'train': 'images/train',
    'val': 'images/val',
    "names": classes,
    "nc": len(classes)
}

In [7]:
with open("train_data.yaml", "w") as f:
    yaml.safe_dump(metadata, f)

yaml_path = Path("train_data.yaml")

In [9]:
model = YOLO("yolov8n.pt")

In [10]:
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    workers=4,
    device=0,
    project="runs/detect",
    name="traffic_vehicle_model"
)

Ultralytics 8.4.163 🚀 Python-3.10.12 torch-2.14.0+cu130 CUDA:0 (NVIDIA L40S, 45458MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=train_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=traffic_vehicle_model, nbs=64, nms=None, opset=None, opt

In [16]:
weights_dir = Path("../model/runs/detect/runs/detect/cherry_eye_model/weights/")
print(list(weights_dir.iterdir()))

[PosixPath('../model/runs/detect/runs/detect/traffic_vehicle_model/weights/last.pt'), PosixPath('../model/runs/detect/runs/detect/traffic_vehicle_model/weights/best.pt')]


In [17]:
shutil.copy(
    "../model/runs/detect/runs/detect/cherry_eye_model/weights/best.pt",
    "../model/model/cherry_eye_model_best.pt"
)

'../model/model/cherry_eye_model_best.pt'

In [18]:
new_model = YOLO("../model/model/cherry_eye_model_best.pt")

In [23]:
model_dir = Path("../model")

In [26]:
test_path = Path(model_dir, "test_data", "test_img")

In [31]:
test_img = list(test_path.iterdir())[55]

In [32]:
results = new_model.predict(
    source= test_img,
    save=True,
    conf=0.25
)


image 1/1 /home/shadeform/model/../model/test_data/test_img/014ff8fa28fc43aa.jpeg: 448x640 3 persons, 1 door, 6.2ms
Speed: 3.2ms preprocess, 6.2ms inference, 1.6ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /home/shadeform/model/runs/detect/predict
